In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    make_scorer,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

DATA_DIR_CANDIDATES = (Path("acp740"), Path("acp_740"), Path("."))
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / "acp740_dataset_with_features.csv").is_file()),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Could not locate the acp740 data directory.")

DATA_PATH = DATA_DIR / "acp740_dataset_with_features.csv"
RANDOM_STATE = 42
combined_dataset = pd.read_csv(DATA_PATH)


## Standardization and statistical feature selection

In [ ]:
features_to_scale = combined_dataset.drop(columns=["data", "label"])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_scale)
scaled_df = pd.DataFrame(scaled_features, columns=features_to_scale.columns)
scaled_df["data"] = combined_dataset["data"].values
scaled_df["label"] = combined_dataset["label"].values
combined_dataset = scaled_df[["data", "label", *features_to_scale.columns]]

group_0 = combined_dataset.loc[combined_dataset["label"] == 0].drop(columns=["data", "label"])
group_1 = combined_dataset.loc[combined_dataset["label"] == 1].drop(columns=["data", "label"])
t_stats, p_values = ttest_ind(group_0, group_1, equal_var=False)
ttest_results = pd.DataFrame(
    {"feature": group_0.columns, "t_stat": t_stats, "p_value": p_values}
)

nan_features = ttest_results.loc[ttest_results["p_value"].isna(), "feature"]
cols_to_drop = [column for column in nan_features if column in combined_dataset.columns]
combined_dataset = combined_dataset.drop(columns=cols_to_drop)
ttest_results = ttest_results.dropna()

p_values_df = ttest_results[["feature", "p_value"]].copy()
_, q_values, _, _ = multipletests(p_values_df["p_value"], method="fdr_bh")
p_values_df["q_value"] = q_values
significant_features = p_values_df.loc[p_values_df["q_value"] < 0.01, "feature"].tolist()
selected_columns = ["data", "label", *significant_features]
new_cleaned_data = combined_dataset[selected_columns]


## Soft voting ensemble

In [ ]:
X = new_cleaned_data.drop(columns=["data", "label"])
y = new_cleaned_data["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
svm_model = SVC(
    kernel="rbf",
    C=1.0,
    gamma="scale",
    probability=True,
    shrinking=True,
    random_state=RANDOM_STATE,
)
ensemble = VotingClassifier(
    estimators=[("random_forest", rf_model), ("svm", svm_model)],
    voting="soft",
    n_jobs=-1,
)
scoring = {
    "accuracy": "accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "mcc": make_scorer(matthews_corrcoef),
}
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
cv_results = cross_validate(
    ensemble,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1,
)

def mean_std(scores):
    return f"{np.mean(scores):.4f} +/- {np.std(scores):.4f}"

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print("\n10-Fold CV Results (Train Set - Soft Voting Ensemble):")
print(f"Accuracy     : {mean_std(cv_results['test_accuracy'])}")
print(f"F1 Score     : {mean_std(cv_results['test_f1'])}")
print(f"Sensitivity  : {mean_std(cv_results['test_recall'])}")
print(f"Precision    : {mean_std(cv_results['test_precision'])}")
print(f"AUC (ROC)    : {mean_std(cv_results['test_roc_auc'])}")
print(f"MCC          : {mean_std(cv_results['test_mcc'])}")

ensemble.fit(X_train, y_train)
y_pred = ensemble.predict(X_test)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("\nFinal Test Set Results (Soft Voting Ensemble):")
print(f"Accuracy     : {(tp + tn) / (tp + tn + fp + fn):.4f}")
print(f"F1 Score     : {2 * tp / (2 * tp + fp + fn):.4f}")
print(f"Sensitivity  : {tp / (tp + fn):.4f}")
print(f"Specificity  : {tn / (tn + fp):.4f}")
print(f"Precision    : {tp / (tp + fp):.4f}")
print(f"AUC (ROC)    : {roc_auc_score(y_test, y_pred):.4f}")
print(f"MCC          : {matthews_corrcoef(y_test, y_pred):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")
